In [0]:
%pip install -q azure-eventhub psycopg2-binary pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("PGHOST", "cdc-pg-chanderpal.postgres.database.azure.com")
dbutils.widgets.text("PGUSER", "")
dbutils.widgets.text("PGPASSWORD", "")
dbutils.widgets.text("EVENTHUB_CONNECTION_STRING", "")
dbutils.widgets.text("EVENTHUB_NAME", "customer-changes")

In [0]:
import json, uuid
import pandas as pd
import psycopg2
from datetime import datetime, timedelta, timezone
from azure.eventhub import EventHubConsumerClient, TransportType

PG = dict(host=dbutils.widgets.get("PGHOST"), port="5432", dbname="postgres",
          user=dbutils.widgets.get("PGUSER"), password=dbutils.widgets.get("PGPASSWORD"), sslmode="require")
EH_CONN = dbutils.widgets.get("EVENTHUB_CONNECTION_STRING")
EH_NAME = dbutils.widgets.get("EVENTHUB_NAME")
assert PG["user"] and PG["password"] and EH_CONN, "Fill in all the widgets first"

def run_sql(sql, params=None, fetch=False):
    conn = psycopg2.connect(**PG)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            result = None
            if fetch:
                cols = [d[0] for d in cur.description]
                result = pd.DataFrame(cur.fetchall(), columns=cols)
        conn.commit()
        return result
    finally:
        conn.close()

print("Baseline:")
display(run_sql("SELECT * FROM customers ORDER BY customer_id", fetch=True).to_string())

run_tag = uuid.uuid4().hex[:8]
a_id = int(run_sql("INSERT INTO customers (full_name, email) VALUES ('Test Delete', %s) RETURNING customer_id",
                   (f"delete.{run_tag}@example.com",), fetch=True).iloc[0, 0])
run_sql("UPDATE customers SET status='INACTIVE' WHERE customer_id=%s", (a_id,))
run_sql("DELETE FROM customers WHERE customer_id=%s", (a_id,))
b_id = int(run_sql("INSERT INTO customers (full_name, email) VALUES ('Test Keep', %s) RETURNING customer_id",
                   (f"keep.{run_tag}@example.com",), fetch=True).iloc[0, 0])
run_sql("UPDATE customers SET status='SUSPENDED' WHERE customer_id=%s", (b_id,))
test_ids = {a_id, b_id}
print("Test customer ids:", sorted(test_ids))

Baseline:


'Empty DataFrame\nColumns: [customer_id, full_name, email, status, updated_at]\nIndex: []'

Test customer ids: [1, 2]


In [0]:
import threading

received = []


def on_event_batch(partition_context, events):
    for ev in events:
        received.append(json.loads(ev.body_as_str()))


consumer = EventHubConsumerClient.from_connection_string(
    EH_CONN, consumer_group="$Default", eventhub_name=EH_NAME,
    transport_type=TransportType.AmqpOverWebsocket,
)
start_time = datetime.now(timezone.utc) - timedelta(minutes=30)


def run_receiver():
    with consumer:
        consumer.receive_batch(on_event_batch=on_event_batch, partition_id="0",
                               starting_position=start_time, max_batch_size=100, max_wait_time=15)


t = threading.Thread(target=run_receiver, daemon=True)
t.start()
t.join(timeout=60)  # never wait more than 60 seconds
if t.is_alive():
    consumer.close()  # stop the receiver if it is still running
    print("Stopped the read after 60 seconds")

test_events = [e for e in received if e["customer_id"] in test_ids]
print(f"Events read: {len(received)} total, {len(test_events)} for the test customers")

events_df = pd.DataFrame([{
    "lsn": int(e["lsn"]), "op": e["op"], "customer_id": e["customer_id"],
    "status_before": (e["before"] or {}).get("status"),
    "status_after": (e["after"] or {}).get("status"),
} for e in test_events]).sort_values("lsn").reset_index(drop=True)
display(events_df)

state = {}
for e in sorted(test_events, key=lambda e: int(e["lsn"])):
    if e["op"] == "D":
        state.pop(e["customer_id"], None)
    else:
        state[e["customer_id"]] = e["after"]
COLUMNS = ("customer_id", "full_name", "email", "status")
rebuilt = {cid: {k: row[k] for k in COLUMNS} for cid, row in state.items()}

db_rows = run_sql("SELECT customer_id, full_name, email, status FROM customers WHERE customer_id = ANY(%s)",
                  (list(test_ids),), fetch=True)
db_state = {int(r["customer_id"]): {"customer_id": int(r["customer_id"]), "full_name": r["full_name"],
            "email": r["email"], "status": r["status"]} for _, r in db_rows.iterrows()}
print("Rebuilt from events:", rebuilt)
print("Database:           ", db_state)
print("MATCH" if rebuilt == db_state else "MISMATCH")

Stopped the read after 60 seconds
Events read: 5 total, 5 for the test customers


lsn,op,customer_id,status_before,status_after
3103786184,I,1,null,ACTIVE
3103790560,U,1,ACTIVE,INACTIVE
3103790808,D,1,INACTIVE,null
3103790984,I,2,null,ACTIVE
3103791352,U,2,ACTIVE,SUSPENDED


Rebuilt from events: {2: {'customer_id': 2, 'full_name': 'Test Keep', 'email': 'keep.1d035361@example.com', 'status': 'SUSPENDED'}}
Database:            {2: {'customer_id': 2, 'full_name': 'Test Keep', 'email': 'keep.1d035361@example.com', 'status': 'SUSPENDED'}}
MATCH


In [0]:
with EventHubConsumerClient.from_connection_string(
    EH_CONN, consumer_group="$Default", eventhub_name=EH_NAME,
    transport_type=TransportType.AmqpOverWebsocket,
) as c:
    print(c.get_eventhub_properties())

{'eventhub_name': 'customer-changes', 'created_at': datetime.datetime(2026, 11, 8, 12, 46, 56, 870000, tzinfo=datetime.timezone.utc), 'partition_ids': ['0']}
